# Classification: will the asset go up or down over the next H days?

**Headline:** *"the model gives an X % probability that it goes up"* at H = 21 days.
It complements the regression notebook (`modelo_retorno_riesgo.ipynb`): same data, same features, same walk-forward validation.

**The classification trap:** markets go up more often than they go down (~58 % of months). A model that always says
"up" is already right 58 % of the time. So accuracy alone is useless: it has to beat that **baseline** in
**AUC** (does it rank well which cases are more likely to go up?) and in **Brier** (are the probabilities right?).

Everything with **NumPy + pandas** (rule in `model/README.md`: no ML libraries). Logistic regression is fitted with Newton-Raphson.

## 1. Setup and access to real data

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import erf, sqrt
warnings.filterwarnings("ignore")

H          = 21        # horizon in days (bars of each asset)
LAMBDA     = 0.97      # EWMA decay used to scale the features
L2         = 1.0       # L2 penalty of the logistic regression
TEST_FROM  = "2000-01-01"   # before 2000 there is almost only ^GSPC, AAPL and MSFT

# Only access to the repo: data/fetch.py (cached prices). Searched upwards from this folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fetch.py").exists()), None)
if ROOT is None:
    raise ImportError("Open this notebook inside the OxHack2026 repo (it needs data/fetch.py).")
sys.path.insert(0, str(ROOT))
from data.assets import all_symbols, label
from data.fetch import get_close, fetch_many, load
print("Repo:", ROOT)

In [ ]:
SYMBOLS = all_symbols()
missing = [s for s in SYMBOLS if load(s) is None]
if missing:
    print("Downloading (once):", missing)
    fetch_many(missing)

closes = {}
for s in SYMBOLS:
    try:
        c = get_close(s).dropna(); c = c[c > 0]
        if len(c) > 500:
            closes[s] = c
        else:
            print("  skipped (short history):", s)
    except Exception as e:
        print("  skipped", s, "->", type(e).__name__, e)
print(f"{len(closes)} assets with data")

## 2. Features and target
Same features as the regression, scaled by EWMA volatility. The target is binary:
`up = 1` if `log(P_{t+H}/P_t) > 0`, otherwise `0`.

In [ ]:
def ewma_vol(r, lam=LAMBDA):
    return np.sqrt(r.pow(2).ewm(alpha=1-lam, adjust=False, min_periods=30).mean())

def build(s, c):
    r = np.log(c).diff().dropna()
    v = ewma_vol(r)
    f = pd.DataFrame(index=r.index)
    f["lag1"]   = r / v
    for k in (5, 21, 63, 126, 252):
        f[f"mom{k}"] = r.rolling(k).sum() / (v*np.sqrt(k))
    f["lvol"]   = np.log(v / r.rolling(63).std())
    cum = r.cumsum(); f["dd63"] = (cum - cum.rolling(63).max()) / (v*np.sqrt(63))
    f["y"]      = np.log(c.shift(-H) / c).reindex(r.index)
    f["sigma"]  = v * np.sqrt(H)
    f["asset"]  = s
    return f

FEATS = ["lag1","mom5","mom21","mom63","mom126","lvol","dd63"]   # logistic with 7 features
TREND = ["mom21","mom63","mom126","mom252"]                      # trend logistic
panel = pd.concat([build(s, c) for s, c in closes.items()]); panel.index.name = "date"
panel[FEATS + ["mom252"]] = panel[FEATS + ["mom252"]].clip(-5, 5)
dataset = panel.dropna().reset_index().sort_values(["date","asset"]).reset_index(drop=True)
dataset["up"] = (dataset["y"] > 0).astype(float)
dataset["z"]  = dataset["y"] / dataset["sigma"]
print(dataset.shape, dataset.date.min().date(), "→", dataset.date.max().date())
print(f"Share of periods that go up: {dataset.up.mean():.1%}")

## 3. Time split (walk-forward with *purge*)
Same as the regression: test from `TEST_FROM` in 6 blocks, each predicted with a model trained only on the past,
with a `2H`-day gap between training and test.

In [ ]:
dates = np.sort(dataset.date.unique())
N_SPLITS, GAP = 6, pd.Timedelta(days=2*H+2)

def walk_forward():
    n = len(dates); start = np.searchsorted(dates, np.datetime64(TEST_FROM)); step = (n-start)//N_SPLITS
    for k in range(N_SPLITS):
        a = start + k*step; b = a + step if k < N_SPLITS-1 else n
        yield (dataset.date < dates[a] - GAP), (dataset.date >= dates[a]) & (dataset.date <= dates[b-1])

for i, (mtr, mte) in enumerate(walk_forward()):
    print(f"fold {i}: train until {dataset.date[mtr].max().date()} ({mtr.sum():,}) | "
          f"test {dataset.date[mte].min().date()}→{dataset.date[mte].max().date()} ({mte.sum():,})")

## 4. Models (pure NumPy)
- **Baseline**: the historical share of up moves in training (always the same probability).
- **Trend logistic**: `P(up) = 1 / (1 + e^{−(w₀ + w·x)})` with the 4 momentum features.
- **7-feature logistic**: the same with the Ridge features.
- **Implied by the regression**: the trend model of the regression notebook gives `mu` and `σ`; with the normal,
  `P(up) = Φ(mu/σ)`. It shows whether classifying directly adds anything over the regression.

In [ ]:
Phi = np.vectorize(lambda x: 0.5*(1 + erf(x/sqrt(2))))

def logit_fit(X, y, l2=L2, iters=50):
    """Logistic regression with L2 penalty (intercept not penalised), via Newton-Raphson."""
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Xs = np.c_[np.ones(len(X)), (X - mu) / sd]
    pen = np.r_[0, np.ones(X.shape[1])] * l2
    w = np.zeros(Xs.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-Xs @ w))
        grad = Xs.T @ (p - y) + pen * w
        hess = (Xs * (p*(1-p))[:, None]).T @ Xs + np.diag(pen)
        step = np.linalg.solve(hess, grad); w -= step
        if np.abs(step).max() < 1e-8:
            break
    return dict(mu=mu, sd=sd, w=w)

def logit_predict(m, X):
    return 1 / (1 + np.exp(-(np.c_[np.ones(len(X)), (X - m["mu"]) / m["sd"]] @ m["w"])))

def trend_fit(X, z):
    s = X.mean(1); b = np.cov(s, z)[0, 1] / s.var()
    return dict(a=z.mean() - b*s.mean(), b=b)

MODELS = ["Baseline", "Trend logistic", "7-feature logistic", "Implied by the regression"]
P = {k: np.full(len(dataset), np.nan) for k in MODELS}
test_mask = np.zeros(len(dataset), bool)
for mtr, mte in walk_forward():
    tr, te = dataset[mtr], dataset[mte]
    P["Baseline"][mte]           = tr["up"].mean()
    P["Trend logistic"][mte]     = logit_predict(logit_fit(tr[TREND].values, tr["up"].values), te[TREND].values)
    P["7-feature logistic"][mte] = logit_predict(logit_fit(tr[FEATS].values, tr["up"].values), te[FEATS].values)
    t = trend_fit(tr[TREND].values, tr["z"].values)
    P["Implied by the regression"][mte] = Phi(t["a"] + t["b"] * te[TREND].values.mean(1))
    test_mask |= mte.values

oos = dataset[test_mask].copy()
for k in MODELS:
    oos[k] = P[k][test_mask]
print("Out-of-sample observations:", len(oos))

## 5. Evaluation
- **Accuracy**: % correct when predicting "up" if `P > 0.5`. Compare against the Baseline, not against 50 %.
- **% predicts down** and **precision on down**: how often the model dares to say "down" and how many of those are right.
- **AUC**: probability that the model gives a higher `P(up)` to a case that went up than to one that went down. 0.5 = chance.
- **Brier**: squared error of the probability (lower is better). **Brier skill** = improvement over the Baseline (positive = better).
- **% years beating Baseline**: in how many years the model's Brier was lower than the Baseline's.

In [ ]:
def auc(y, p):
    r = pd.Series(p).rank().values; n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1*(n1+1)/2) / (n1*n0)

def clf_metrics(y, p, p_base):
    pred = p > 0.5
    bs, bs0 = np.mean((p - y)**2), np.mean((p_base - y)**2)
    eps = 1e-12
    return dict(accuracy=(pred == y).mean(), pct_predicts_down=(~pred).mean(),
                precision_down=(y[~pred] == 0).mean() if (~pred).any() else np.nan,
                AUC=auc(y, p), Brier=bs, Brier_skill=1 - bs/bs0,
                log_loss=-np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps)))

def years_beating(g, k):
    a = g.groupby(g.date.dt.year).apply(lambda x: ((x[k] - x.up)**2).mean() < ((x["Baseline"] - x.up)**2).mean())
    return a.mean()

y, pb = oos["up"].values, oos["Baseline"].values
table = pd.DataFrame({k: {**clf_metrics(y, oos[k].values, pb), "years_beating_base": years_beating(oos, k)} for k in MODELS}).T
table.columns = ["Accuracy", "% predicts down", "Precision on down", "AUC", "Brier", "Brier skill", "Log loss", "% years beating Baseline"]
print(f"Actual share of up moves in the test: {y.mean():.1%}")
table.round(4)

In [ ]:
# The chosen model is the one with the highest Brier skill (best out-of-sample probabilities)
BEST = table.drop("Baseline")["Brier skill"].idxmax()
print("Chosen model:", BEST)

rows = {}
for a, g in oos.groupby("asset"):
    m = clf_metrics(g.up.values, g[BEST].values, g["Baseline"].values)
    rows[label(a)] = dict(up_rate=g.up.mean(), accuracy=m["accuracy"], accuracy_base=(g.up == 1).mean(),
                          AUC=m["AUC"], Brier_skill=m["Brier_skill"])
pd.DataFrame(rows).T.sort_values("AUC", ascending=False).round(3)

## 6. Calibration: when it says 60 %, does it go up 60 % of the time?
Predictions are grouped in probability deciles and the mean predicted probability is compared with the actual frequency.
A calibrated model lies on the diagonal.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k in ["Trend logistic", "7-feature logistic", "Implied by the regression"]:
    q = pd.qcut(oos[k], 10, duplicates="drop")
    cal = oos.groupby(q).agg(pred=(k, "mean"), actual=("up", "mean"))
    ax[0].plot(cal.pred, cal.actual, "o-", label=k)
lo, hi = oos[MODELS[1:]].min().min(), oos[MODELS[1:]].max().max()
ax[0].plot([lo, hi], [lo, hi], "k--", lw=1); ax[0].axhline(y.mean(), color="gray", lw=.8, ls=":")
ax[0].set(xlabel="Predicted P(up)", ylabel="Actual frequency of up moves", title="Calibration by decile"); ax[0].legend(fontsize=8)

ax[1].hist(oos[BEST], bins=60, alpha=.8); ax[1].axvline(0.5, color="k", lw=1)
ax[1].set(title=f"Distribution of P(up) — {BEST}", xlabel="P(up)")
plt.tight_layout(); plt.show()

In [ ]:
# Calibration table of the chosen model: if it separates well, the top decile goes up far more than the bottom one
q = pd.qcut(oos[BEST], 10, labels=False, duplicates="drop")
oos.groupby(q).agg(predicted_P=(BEST, "mean"), actual_frequency=("up", "mean"),
                   mean_return_pct=("y", lambda r: r.mean()*100), cases=("up", "size")).round(3)

## 7. Final model and `predict_class` function
The chosen model is retrained on the **whole** history. `predict_class(prices)` returns the probability of an up move over H days,
the class (`"up"`/`"down"`) and the edge over the base rate (how far the model moves away from "the usual").

In [ ]:
BASE_RATE = dataset["up"].mean()
if BEST == "Trend logistic":
    FINAL = logit_fit(dataset[TREND].values, dataset["up"].values); COLS = TREND
    prob = lambda x: float(logit_predict(FINAL, x[None, :])[0])
elif BEST == "7-feature logistic":
    FINAL = logit_fit(dataset[FEATS].values, dataset["up"].values); COLS = FEATS
    prob = lambda x: float(logit_predict(FINAL, x[None, :])[0])
else:
    FINAL = trend_fit(dataset[TREND].values, dataset["z"].values); COLS = TREND
    prob = lambda x: float(Phi(FINAL["a"] + FINAL["b"] * x.mean()))
print("Final model:", BEST)
if "w" in FINAL:
    print(pd.Series(FINAL["w"], index=["intercept"] + COLS).round(4).to_string())

def predict_class(prices: pd.Series) -> dict:
    """prices = daily closes with the date as index. Returns P(up) over H days."""
    c = prices.dropna(); c = c[c > 0]
    f = build("x", c).iloc[-1]                     # last row: only information up to today
    x = np.clip(f[COLS].astype(float).values, -5, 5)
    p = prob(x)
    return {"p_up": p, "class": "up" if p > 0.5 else "down",
            "edge_vs_base": p - BASE_RATE, "horizon_days": H}

## 8. Current forecast: "the model gives an X % probability that it goes up"

In [ ]:
rows = {}
for s, c in closes.items():
    p = predict_class(c)
    rows[s] = dict(asset=label(s), last_date=c.index[-1].date(), p_up=p["p_up"],
                   cls=p["class"], edge_vs_base=p["edge_vs_base"])
forecast = pd.DataFrame(rows).T.sort_values("p_up", ascending=False)
print(f"Historical base rate of up moves over {H} days: {BASE_RATE:.1%}\n")
for s, r in forecast.iterrows():
    print(f"{r.asset:<38} P(up) {r.p_up:.1%}  → {r.cls:<4}  ({r.edge_vs_base*100:+.1f} pp vs base)")

## 9. Export

In [ ]:
out = ROOT / "notebooks" / "outputs"; out.mkdir(parents=True, exist_ok=True)
forecast.assign(horizon_days=H).to_csv(out / "pronostico_clasificacion.csv")
table.to_csv(out / "metricas_clasificacion_oos.csv")
print("Saved to", out)

### Notes
- **Do not look at accuracy alone.** Since ~58 % of periods go up, a model that almost always says "up" has high accuracy without knowing anything. The honest comparison is against the Baseline in AUC, Brier skill and % of years.
- **The model will almost never say "down".** The upward drift means `P(up)` rarely falls below 50 %. What is useful is the **difference from the base rate**: an asset at 52 % has a worse outlook than one at 65 %, even though both say "up".
- **Relation to the regression:** `P(loss)` in the regression notebook is `1 − P(up)` of the "Implied by the regression" row. If the logistic does not beat it, it is better to use a single model (the regression) for return, risk and probability.
- **Mixed calendars:** H counts bars of each asset (≈ 1 month for stocks, 3 weeks for crypto).